# 🤖 Training & Fine-tuning LLMs — Practice Notebook

> **Run this notebook on Google Colab** → Runtime → Change runtime type → **T4 GPU**

---

## 📋 Table of Contents
| Problem | Topic | Difficulty | Time |
|---------|-------|-----------|------|
| [P1](#p1) | Dataset Preparation & Pre-processing | 🟢 Easy | 30 min |
| [P2](#p2) | Fine-tune GPT-2 on Indian Food Recipes | 🟡 Medium | 60 min |
| [P3](#p3) | Hyperparameter Sensitivity Experiment | 🟡 Medium | 45 min |
| [P4](#p4) | Compute BLEU & ROUGE Scores | 🟢 Easy | 25 min |
| [P5](#p5) | Build a RAG Pipeline from Scratch | 🔴 Hard | 60 min |

---
**Requirements:** All datasets are free from HuggingFace Hub. No API keys needed.


## ⚙️ Install All Dependencies (Run Once)


In [ ]:
# Install everything needed for all 5 problems
!pip install transformers datasets evaluate accelerate \
             rouge-score sacrebleu sentence-transformers \
             faiss-cpu matplotlib wordcloud scikit-learn -q
print("✅ All packages installed!")

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

---
<a id='p1'></a>
# 🟢 Problem 1 — Dataset Preparation & Pre-processing
**Dataset:** `knkarthick/dialogsum` | **Time:** ~30 min | **GPU:** Not required

### 🎯 Objectives
- Load and inspect a real HuggingFace dialogue dataset
- Clean, filter and normalize text
- Tokenize text for a Flan-T5 transformer model
- Visualize token length distributions

### 💡 Key Concept
> **Garbage in, garbage out.** A model is only as good as its training data. Investing in data quality pays off more than any hyperparameter tweak.


In [ ]:
# P1 — Cell 1: Load the DialogSum dataset
from datasets import load_dataset

dataset = load_dataset("knkarthick/dialogsum")
print(dataset)
print("\n--- Sample entry ---")
print(dataset["train"][0])

In [ ]:
# P1 — Cell 2: Explore the data structure
import pandas as pd

df = dataset["train"].to_pandas()
print(f"Training samples : {len(df)}")
print(f"Columns          : {df.columns.tolist()}")
print(f"\nSample dialogue:\n{df['dialogue'][0][:300]}...")
print(f"\nSample summary:\n{df['summary'][0]}")

In [ ]:
# P1 — Cell 3: Data cleaning
import re

def clean_text(text):
    text = text.strip()
    # Remove speaker labels like "#Person1#:"
    text = re.sub(r'#\w+#:\s*', '', text)
    # Collapse multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text)
    # Drop very short texts (less than 20 chars — likely corrupted)
    return text if len(text) > 20 else None

df['clean_dialogue'] = df['dialogue'].apply(clean_text)
df['clean_summary']  = df['summary'].apply(clean_text)

# Drop rows where cleaning returned None
before = len(df)
df = df.dropna(subset=['clean_dialogue', 'clean_summary'])
print(f"Before cleaning : {before} samples")
print(f"After cleaning  : {len(df)} samples")
print(f"Removed         : {before - len(df)} samples")

print("\n--- Cleaned example ---")
print("Dialogue:", df['clean_dialogue'].iloc[0][:200])
print("Summary :", df['clean_summary'].iloc[0])

In [ ]:
# P1 — Cell 4: Tokenize with Flan-T5 tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

def tokenize_example(row):
    prompt   = f"Summarize this dialogue:\n{row['clean_dialogue']}"
    response = row['clean_summary']
    input_tokens  = tokenizer(prompt,   truncation=True, max_length=512)
    output_tokens = tokenizer(response, truncation=True, max_length=128)
    return {
        'input_len' : len(input_tokens['input_ids']),
        'output_len': len(output_tokens['input_ids'])
    }

print("Tokenizing... (may take ~1 minute)")
df[['input_len', 'output_len']] = df.apply(tokenize_example, axis=1, result_type='expand')

print(f"\nDialogue tokens  — mean: {df['input_len'].mean():.1f}, max: {df['input_len'].max()}")
print(f"Summary tokens   — mean: {df['output_len'].mean():.1f}, max: {df['output_len'].max()}")
print(f"Samples exceeding 512 tokens: {(df['input_len'] >= 512).sum()} ({(df['input_len'] >= 512).mean()*100:.1f}%)")

In [ ]:
# P1 — Cell 5: Visualize token distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('P1 — Token Length Distributions', fontsize=14, fontweight='bold')

axes[0].hist(df['input_len'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(512, color='red', linestyle='--', label='max_length=512')
axes[0].set_title('Dialogue (Input) Token Lengths')
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(df['output_len'], bins=40, color='tomato', edgecolor='white', alpha=0.85)
axes[1].axvline(128, color='navy', linestyle='--', label='max_length=128')
axes[1].set_title('Summary (Output) Token Lengths')
axes[1].set_xlabel('Number of Tokens')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✅ P1 Complete! Key takeaways:")
print("   • Most dialogues fit within 512 tokens — good for Flan-T5")
print("   • Summaries are short (~30 tokens) — perfect for max_length=128")

In [ ]:
# P1 — 🔥 Stretch Challenge: Data Augmentation with Synonym Replacement
# NLTK WordNet synonym replacement
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
from nltk.corpus import wordnet
import random

def synonym_replace(text, n=2):
    """Replace n random words with synonyms."""
    words = text.split()
    new_words = words.copy()
    replaced = 0
    indices = random.sample(range(len(words)), min(n * 3, len(words)))
    for i in indices:
        syns = wordnet.synsets(words[i])
        if syns and replaced < n:
            syn_words = [l.name().replace('_', ' ') for l in syns[0].lemmas() if l.name() != words[i]]
            if syn_words:
                new_words[i] = random.choice(syn_words)
                replaced += 1
    return ' '.join(new_words)

# Demo on 3 examples
print("📝 Synonym Replacement Examples:\n")
for i in range(3):
    original  = df['clean_summary'].iloc[i]
    augmented = synonym_replace(original)
    print(f"Original : {original}")
    print(f"Augmented: {augmented}")
    print()

---
<a id='p2'></a>
# 🟡 Problem 2 — Fine-tune GPT-2 on Indian Food Recipes
**Dataset:** `Hieu-Pham/indian_food_recipes` | **Time:** ~60 min | **GPU:** Required (T4)

### 🎯 Objectives
- Fine-tune GPT-2 on a custom domain dataset
- Set up TrainingArguments with key hyperparameters
- Generate text from base vs fine-tuned model
- Observe how the model's style shifts after training

### 💡 Key Concept
> GPT-2 knows nothing about Indian food. Fine-tuning "teaches" it the vocabulary, structure, and format of recipes — without changing its underlying language ability.


In [ ]:
# P2 — Cell 1: Load tokenizer and base model, check GPU
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

if not torch.cuda.is_available():
    print("⚠️  WARNING: No GPU detected! This will be very slow.")
    print("   Go to: Runtime → Change runtime type → T4 GPU")

model_name = "gpt2"
tokenizer  = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no pad token by default

print(f"✅ Tokenizer loaded: vocab size = {tokenizer.vocab_size}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# P2 — Cell 2: Load and format the recipe dataset
from datasets import load_dataset

recipe_data = load_dataset("Hieu-Pham/indian_food_recipes", split="train")
print(f"Recipes loaded: {len(recipe_data)}")
print(f"Columns       : {recipe_data.column_names}")
print("\nSample:")
print(recipe_data[0])

In [ ]:
# P2 — Cell 3: Format recipes and save to text file
def format_recipe(example):
    # Build a structured recipe string GPT-2 can learn from
    name  = example.get('name', example.get('TranslatedRecipeName', 'Unknown'))
    ingr  = example.get('ingredients', example.get('TranslatedIngredients', ''))
    instr = example.get('instructions', example.get('TranslatedInstructions', ''))
    text  = f"Recipe: {name}\nIngredients: {ingr}\nInstructions: {instr}\n<|endoftext|>"
    return {"text": text}

recipe_data = recipe_data.map(format_recipe)

# Save to text file (required by TextDataset)
with open("recipes.txt", "w", encoding="utf-8") as f:
    for item in recipe_data:
        f.write(item["text"] + "\n")

print(f"✅ Saved {len(recipe_data)} recipes to recipes.txt")
print("\nSample formatted recipe:")
print(recipe_data["text"][0][:300])

In [ ]:
# P2 — Cell 4: Create token dataset for causal language modeling
from transformers import TextDataset, DataCollatorForLanguageModeling

train_dataset = TextDataset(
    tokenizer  = tokenizer,
    file_path  = "recipes.txt",
    block_size = 256  # Each training block = 256 tokens
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False  # GPT-2 uses causal LM (predict next token), NOT masked LM
)

print(f"Training blocks created: {len(train_dataset)}")
print("Each block = 256 tokens from the recipe file")

In [ ]:
# P2 — Cell 5: Generate text BEFORE fine-tuning (baseline)
import textwrap

base_model = GPT2LMHeadModel.from_pretrained(model_name)

def generate_text(model, prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            do_sample         = True,
            temperature       = 0.85,
            top_p             = 0.92,
            pad_token_id      = tokenizer.eos_token_id
        )
    return textwrap.fill(
        tokenizer.decode(output[0], skip_special_tokens=True), 90
    )

prompt = "Recipe: Butter Chicken\nIngredients:"
print("=" * 60)
print("BASE GPT-2 (no fine-tuning):")
print("=" * 60)
print(generate_text(base_model, prompt))
print("\n👆 Notice: GPT-2 doesn't know what a recipe looks like!")

In [ ]:
# P2 — Cell 6: Fine-tune GPT-2
# ⏱ Expected time: ~30-45 min on T4 GPU
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

ft_model = GPT2LMHeadModel.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir                  = "./gpt2-recipes",
    num_train_epochs            = 3,        # Try 5 for even better results
    per_device_train_batch_size = 4,
    learning_rate               = 5e-5,     # Standard LR for LLM fine-tuning
    warmup_steps                = 100,      # Gradual LR ramp-up
    weight_decay                = 0.01,     # Regularization to prevent overfitting
    logging_steps               = 50,
    save_steps                  = 500,
    save_total_limit            = 1,
    prediction_loss_only        = True,
    fp16                        = torch.cuda.is_available(),  # Mixed precision on GPU
    report_to                   = "none"
)

trainer = Trainer(
    model         = ft_model,
    args          = training_args,
    data_collator = data_collator,
    train_dataset = train_dataset,
)

print("🚀 Starting fine-tuning...")
print(f"   Epochs         : {training_args.num_train_epochs}")
print(f"   Batch size     : {training_args.per_device_train_batch_size}")
print(f"   Learning rate  : {training_args.learning_rate}")
print(f"   Warmup steps   : {training_args.warmup_steps}")
print()

train_result = trainer.train()
print("\n✅ Fine-tuning complete!")
print(f"   Final training loss: {train_result.training_loss:.4f}")

In [ ]:
# P2 — Cell 7: Compare BASE vs FINE-TUNED outputs
dishes = ["Butter Chicken", "Biryani", "Samosa", "Dal Makhani", "Palak Paneer"]

print("=" * 70)
print("BEFORE vs AFTER FINE-TUNING COMPARISON")
print("=" * 70)

for dish in dishes:
    prompt = f"Recipe: {dish}\nIngredients:"
    print(f"\n🍛 Prompt: {prompt}")
    print(f"\n❌ Base GPT-2:")
    print(generate_text(base_model, prompt, max_new_tokens=80))
    print(f"\n✅ Fine-tuned GPT-2:")
    print(generate_text(ft_model, prompt, max_new_tokens=80))
    print("-" * 70)

In [ ]:
# P2 — Cell 8: Plot the training loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps  = [x['step'] for x in log_history if 'loss' in x]
losses = [x['loss'] for x in log_history if 'loss' in x]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, color='steelblue', linewidth=2)
plt.fill_between(steps, losses, alpha=0.15, color='steelblue')
plt.title('P2 — GPT-2 Fine-tuning Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Starting loss : {losses[0]:.4f}")
print(f"Final loss    : {losses[-1]:.4f}")
print(f"Improvement   : {losses[0] - losses[-1]:.4f} ({(losses[0]-losses[-1])/losses[0]*100:.1f}%)")

---
<a id='p3'></a>
# 🟡 Problem 3 — Hyperparameter Sensitivity Experiment
**Dataset:** `glue/sst2` (sentiment) | **Time:** ~45 min | **GPU:** Recommended

### 🎯 Objectives
- Run DistilBERT with 4 different learning rates
- Plot accuracy vs learning rate — see the "Goldilocks zone"
- Understand the practical effect of warmup steps
- Learn why 5e-5 is the standard fine-tuning LR for LLMs

### 💡 Key Concept
> **Learning rate is the most important hyperparameter.** Too high = the model "forgets" its pre-training instantly. Too low = it barely changes at all.


In [ ]:
# P3 — Cell 1: Load SST-2 sentiment dataset
from datasets import load_dataset
from transformers import AutoTokenizer

dataset   = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["sentence"], truncation=True, padding="max_length", max_length=128
    )

# Use a small subset for speed (500 train / 200 val)
small_train = dataset["train"].shuffle(seed=42).select(range(500)).map(tokenize, batched=True)
small_val   = dataset["validation"].shuffle(seed=42).select(range(200)).map(tokenize, batched=True)

small_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
small_val.set_format("torch",   columns=["input_ids", "attention_mask", "label"])

print(f"Train samples : {len(small_train)}")
print(f"Val samples   : {len(small_val)}")
print(f"Example label : {small_train[0]['label'].item()} (0=negative, 1=positive)")

In [ ]:
# P3 — Cell 2: Run experiment across 4 learning rates
# ⏱ Expected time: ~15-20 min on T4 GPU
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

learning_rates = [1e-3, 1e-4, 5e-5, 1e-6]  # Too high → Just right → Too low
results = {}

for lr in learning_rates:
    print(f"\n▶ Training with LR = {lr}")
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    args = TrainingArguments(
        output_dir                      = f"./lr-{lr}",
        num_train_epochs                = 3,
        per_device_train_batch_size     = 16,
        per_device_eval_batch_size      = 32,
        learning_rate                   = lr,
        warmup_steps                    = 50,
        evaluation_strategy             = "epoch",
        logging_steps                   = 20,
        load_best_model_at_end          = True,
        metric_for_best_model           = "accuracy",
        report_to                       = "none",
        save_strategy                   = "epoch"
    )
    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = small_train,
        eval_dataset    = small_val,
        compute_metrics = compute_metrics,
    )
    trainer.train()
    eval_result = trainer.evaluate()
    results[lr] = {
        "accuracy"    : eval_result["eval_accuracy"],
        "loss"        : eval_result["eval_loss"],
        "log_history" : trainer.state.log_history
    }
    print(f"   → Val accuracy: {eval_result['eval_accuracy']:.4f} | Val loss: {eval_result['eval_loss']:.4f}")

print("\n✅ All experiments done!")

In [ ]:
# P3 — Cell 3: Visualize results
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 5))
fig.suptitle('P3 — Hyperparameter Sensitivity: Learning Rate', fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# Bar chart: final accuracy per LR
ax1 = fig.add_subplot(gs[0])
lrs_str = [f"{lr:.0e}" for lr in learning_rates]
accs    = [results[lr]["accuracy"] for lr in learning_rates]
colors  = ['#e05252', '#f0a050', '#4caf80', '#7090c8']
bars = ax1.bar(lrs_str, accs, color=colors, edgecolor='white', width=0.5)
ax1.set_xlabel("Learning Rate", fontsize=11)
ax1.set_ylabel("Validation Accuracy", fontsize=11)
ax1.set_title("Final Accuracy vs Learning Rate")
ax1.set_ylim(0.45, 1.0)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.008,
             f"{val:.3f}", ha='center', fontsize=10, fontweight='bold')
ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.6, label='Random baseline')
ax1.legend()

# Table
ax2 = fig.add_subplot(gs[1])
ax2.axis('off')
labels_col = ["LR=1e-3\n(Too High)", "LR=1e-4", "LR=5e-5\n(Sweet Spot)", "LR=1e-6\n(Too Low)"]
table_data = [
    [lbl, f"{results[lr]['accuracy']:.4f}", f"{results[lr]['loss']:.4f}"]
    for lr, lbl in zip(learning_rates, labels_col)
]
tbl = ax2.table(
    cellText  = table_data,
    colLabels = ["Config", "Val Accuracy", "Val Loss"],
    loc='center', cellLoc='center'
)
tbl.scale(1.2, 2.0)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
ax2.set_title("Results Summary", pad=20)

plt.savefig('p3_lr_results.png', dpi=120, bbox_inches='tight')
plt.show()

best_lr = max(results, key=lambda x: results[x]["accuracy"])
print(f"\n🏆 Best LR: {best_lr} → Accuracy: {results[best_lr]['accuracy']:.4f}")
print("   → 5e-5 is the industry standard for fine-tuning transformers — now you know why!")

In [ ]:
# P3 — 🔥 Stretch Challenge: Compare batch sizes
# Run the same experiment but vary batch_size = [4, 8, 16, 32]
# Keep LR fixed at 5e-5 (the best from above)
# Which batch size gives the best accuracy? Which is fastest?

batch_sizes = [4, 8, 16, 32]
batch_results = {}

import time
for bs in batch_sizes:
    print(f"\n▶ Training with batch_size = {bs}")
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    args = TrainingArguments(
        output_dir                  = f"./bs-{bs}",
        num_train_epochs            = 2,
        per_device_train_batch_size = bs,
        per_device_eval_batch_size  = 32,
        learning_rate               = 5e-5,
        warmup_steps                = 50,
        evaluation_strategy         = "epoch",
        report_to                   = "none",
        save_strategy               = "epoch"
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=small_train, eval_dataset=small_val,
        compute_metrics=compute_metrics
    )
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    ev = trainer.evaluate()
    batch_results[bs] = {"accuracy": ev["eval_accuracy"], "time": elapsed}
    print(f"   accuracy={ev['eval_accuracy']:.4f}  time={elapsed:.0f}s")

print("\n📊 Batch Size Summary:")
for bs, r in batch_results.items():
    print(f"  bs={bs:2d} → accuracy={r['accuracy']:.4f}, time={r['time']:.0f}s")

---
<a id='p4'></a>
# 🟢 Problem 4 — Compute BLEU & ROUGE Scores
**Dataset:** `knkarthick/dialogsum` (test split) | **Time:** ~25 min | **GPU:** Not required

### 🎯 Objectives
- Compute BLEU manually (understand the math)
- Evaluate Flan-T5 summarization using HuggingFace `evaluate`
- Interpret ROUGE-1, ROUGE-2, ROUGE-L scores
- See qualitative examples alongside quantitative scores

### 💡 Key Concept
> **BLEU measures precision** (does the model use the right words?). **ROUGE measures recall** (does the model cover everything in the reference?). You need both for a complete picture.


In [ ]:
# P4 — Cell 1: Understand BLEU manually — no libraries needed
def get_ngrams(text, n):
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]

reference = "The government announced a new climate policy targeting carbon emissions"
generated = "A new government climate policy aims to reduce carbon emissions"

print("📐 Manual BLEU Calculation")
print(f"Reference : {reference}")
print(f"Generated : {generated}")
print()

for n in [1, 2, 3]:
    ref_ng = set(get_ngrams(reference, n))
    gen_ng = get_ngrams(generated, n)
    matches   = sum(1 for ng in gen_ng if ng in ref_ng)
    precision = matches / len(gen_ng) if gen_ng else 0
    print(f"BLEU-{n}:  {matches} / {len(gen_ng)} n-grams matched → precision = {precision:.3f}")

print()
print("📐 Manual ROUGE-1 (Recall) Calculation")
ref_words = set(reference.lower().split())
gen_words = generated.lower().split()
recall_matches = sum(1 for w in gen_words if w in ref_words)
recall = recall_matches / len(reference.lower().split())
print(f"Words from reference covered in generated: {recall_matches}/{len(reference.split())} = {recall:.3f}")

In [ ]:
# P4 — Cell 2: Load test dataset and generate summaries with Flan-T5
from transformers import pipeline
from datasets import load_dataset

test_data = load_dataset("knkarthick/dialogsum", split="test[:50]")
print(f"Test samples: {len(test_data)}")

summarizer = pipeline(
    "text2text-generation",
    model      = "google/flan-t5-base",
    tokenizer  = "google/flan-t5-base",
    max_length = 128
)

print("Generating summaries... (~2-3 min)")
predictions = []
references  = []

for i, item in enumerate(test_data):
    prompt = f"Summarize the following conversation:\n{item['dialogue']}"
    output = summarizer(prompt)[0]['generated_text']
    predictions.append(output)
    references.append(item['summary'])
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/50 done...")

print("✅ All summaries generated!")

In [ ]:
# P4 — Cell 3: Compute BLEU and ROUGE using HuggingFace evaluate
import evaluate

# BLEU
bleu = evaluate.load("sacrebleu")
bleu_result = bleu.compute(
    predictions = predictions,
    references  = [[r] for r in references]  # BLEU expects list of lists
)

# ROUGE
rouge = evaluate.load("rouge")
rouge_result = rouge.compute(
    predictions = predictions,
    references  = references
)

print("=" * 50)
print("📊 EVALUATION RESULTS — Flan-T5-base on DialogSum")
print("=" * 50)
print(f"\nBLEU Score  : {bleu_result['score']:.2f} / 100")
print(f"  (0-20: poor | 20-40: decent | 40-60: good | 60+: near-human)")
print(f"\nROUGE-1     : {rouge_result['rouge1']:.4f}")
print(f"  (unigram recall — individual word overlap)")
print(f"\nROUGE-2     : {rouge_result['rouge2']:.4f}")
print(f"  (bigram recall — two-word phrase overlap)")
print(f"\nROUGE-L     : {rouge_result['rougeL']:.4f}")
print(f"  (longest common subsequence)")

In [ ]:
# P4 — Cell 4: Visualize scores + show qualitative examples
import matplotlib.pyplot as plt

# Plot ROUGE scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('P4 — BLEU & ROUGE Evaluation: Flan-T5 on DialogSum', fontsize=13, fontweight='bold')

rouge_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
rouge_values = [rouge_result['rouge1'], rouge_result['rouge2'], rouge_result['rougeL']]
colors = ['#22d3a0', '#60a5fa', '#a78bfa']

bars = ax1.bar(rouge_labels, rouge_values, color=colors, edgecolor='white', width=0.45)
ax1.set_ylim(0, 1)
ax1.set_title('ROUGE Scores')
ax1.set_ylabel('Score (0-1)')
for bar, val in zip(bars, rouge_values):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f"{val:.3f}", ha='center', fontsize=11, fontweight='bold')

# BLEU as gauge
ax2.barh(['BLEU Score'], [bleu_result['score']], color='#f97316', edgecolor='white', height=0.35)
ax2.barh(['Max (100)'],  [100],                 color='#e2e8f0', edgecolor='white', height=0.35)
ax2.set_xlim(0, 100)
ax2.set_title('BLEU Score')
ax2.text(bleu_result['score'] + 1, 0, f"{bleu_result['score']:.1f}", va='center', fontsize=11, fontweight='bold')
ax2.axvline(20, color='red',   linestyle=':', alpha=0.7, label='20 = Decent')
ax2.axvline(40, color='green', linestyle=':', alpha=0.7, label='40 = Good')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Qualitative examples
print("\n📝 QUALITATIVE EXAMPLES (Reference vs Generated)")
print("=" * 70)
for i in [0, 5, 15, 25]:
    print(f"\n[Example {i+1}]")
    print(f"Dialogue  : {test_data[i]['dialogue'][:120].strip()}...")
    print(f"Reference : {references[i]}")
    print(f"Generated : {predictions[i]}")

---
<a id='p5'></a>
# 🔴 Problem 5 — Build a RAG Pipeline from Scratch
**Knowledge base: AI companies (OpenAI, Google, Microsoft)** | **Time:** ~60 min | **GPU:** Not required

### 🎯 Objectives
- Embed a custom knowledge base into a FAISS vector store
- Retrieve relevant context using semantic similarity
- Augment prompts with retrieved context
- Compare RAG answers vs vanilla LLM answers side-by-side

### 💡 Key Concept
> **RAG = Open-Book Exam.** A vanilla LLM can only answer from memory (training data). RAG lets the model "look up" your documents at query time — no retraining needed, always up-to-date.


In [ ]:
# P5 — Cell 1: Define your knowledge base
# Domain: Major AI companies — OpenAI, Google, Microsoft
# 💡 TIP: Replace or extend these with your own documents!

documents = [
    # OpenAI
    "OpenAI was founded in 2015 as a non-profit AI research lab by Sam Altman, Elon Musk, and others, with the mission to ensure AI benefits all of humanity.",
    "OpenAI released GPT-3 in 2020 with 175 billion parameters, trained on a diverse dataset of internet text called WebText.",
    "ChatGPT was launched by OpenAI in November 2022 and reached 100 million users in just two months, the fastest-growing consumer app in history.",
    "GPT-4, released in March 2023, is a multimodal model that can accept both text and image inputs and outperforms GPT-3.5 on most benchmarks.",
    "OpenAI's DALL-E is a text-to-image generation model that creates images from natural language descriptions.",
    "OpenAI offers its models via an API, with pricing based on tokens — a token is roughly 4 characters or 0.75 words of English text.",
    "OpenAI introduced the concept of RLHF (Reinforcement Learning from Human Feedback) to align LLM outputs with human preferences.",

    # Google / DeepMind
    "Google DeepMind was formed in 2023 by merging Google Brain and DeepMind into a single AI research unit.",
    "Gemini is Google's flagship multimodal AI model family, available in sizes: Nano, Flash, Pro, and Ultra.",
    "Google's BERT (Bidirectional Encoder Representations from Transformers) introduced bidirectional pre-training for NLP tasks in 2018.",
    "Google's Flan-T5 is an instruction-tuned version of the T5 model, fine-tuned on a mixture of tasks to improve zero-shot performance.",
    "Google developed the Transformer architecture in 2017, published in the paper 'Attention Is All You Need' by Vaswani et al.",
    "Google Cloud offers Vertex AI, a managed ML platform that supports training, deploying, and monitoring machine learning models at scale.",
    "Google's AlphaFold 2 predicted the 3D structure of nearly all known proteins, a breakthrough in computational biology.",

    # Microsoft
    "Microsoft invested over $13 billion in OpenAI and integrated GPT-4 into its products including Bing Search and Microsoft 365 Copilot.",
    "Microsoft Azure OpenAI Service gives enterprise customers access to OpenAI models like GPT-4 with added security and compliance features.",
    "GitHub Copilot, built on OpenAI Codex, is an AI pair programmer that suggests code completions directly in code editors like VS Code.",
    "Microsoft's Phi series are small language models (SLMs) designed to run on edge devices with limited compute, achieving high performance with fewer parameters.",
    "Microsoft introduced Copilot in Windows 11 as an AI assistant integrated into the operating system for productivity tasks.",
    "Azure Machine Learning is Microsoft's cloud platform for building, training, and deploying machine learning models at enterprise scale.",
]

print(f"Knowledge base: {len(documents)} documents loaded")
print("\nDocument breakdown:")
print(f"  OpenAI docs    : 7")
print(f"  Google docs    : 7")
print(f"  Microsoft docs : 6")
print("\nSample entries:")
for i, doc in enumerate(documents[:3]):
    print(f"  [{i}] {doc[:90]}...")


In [ ]:
# P5 — Cell 2: Embed all documents into a FAISS vector index
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# all-MiniLM-L6-v2 = fast, lightweight, great quality for semantic search
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding documents...")
doc_embeddings = embedder.encode(documents, convert_to_numpy=True, show_progress_bar=True)
print(f"\nEmbedding shape: {doc_embeddings.shape}")
print(f"Each document → {doc_embeddings.shape[1]}-dimensional vector")

# Build FAISS index (IndexFlatL2 = exact nearest neighbor by Euclidean distance)
dim   = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(doc_embeddings.astype('float32'))

print(f"\n✅ FAISS index built with {index.ntotal} vectors")
print("   Ready for semantic search!")

In [ ]:
# P5 — Cell 3: Test the retriever before connecting to an LLM
def retrieve(query, top_k=3, verbose=True):
    query_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_vec, top_k)
    retrieved = [documents[i] for i in indices[0]]
    if verbose:
        print(f"\n🔍 Query: {query}")
        print(f"Top-{top_k} retrieved documents:")
        for j, (doc, dist) in enumerate(zip(retrieved, distances[0])):
            print(f"  [{j+1}] (dist={dist:.3f}) {doc[:80]}...")
    return retrieved, distances[0]

# Test retrieval on a few queries
test_queries = [
    "What is BNPL and how does PayPal offer it?",
    "How does learning rate affect model training?",
    "What tools are used for vector search in RAG?",
]

for q in test_queries:
    retrieve(q, top_k=2, verbose=True)

In [ ]:
# P5 — Cell 4: Load Flan-T5 as the generator
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model      = "google/flan-t5-base",
    max_length = 256
)
print("✅ Flan-T5-base generator loaded")

In [ ]:
# P5 — Cell 5: The complete RAG function
def rag_answer(query, top_k=3):
    """Full RAG pipeline: Retrieve → Augment → Generate"""
    # Step 1: Retrieve relevant documents
    context_docs, scores = retrieve(query, top_k=top_k, verbose=False)
    context = "\n".join(f"- {doc}" for doc in context_docs)

    # Step 2: Augment the prompt with retrieved context
    prompt = f"""Based on the following context, answer the question accurately.

Context:
{context}

Question: {query}

Answer:"""

    # Step 3: Generate with LLM
    response = generator(prompt)[0]['generated_text']
    return response, context_docs

def vanilla_answer(query):
    """Plain LLM answer — no retrieval"""
    return generator(f"Answer this question: {query}")[0]['generated_text']

print("✅ RAG pipeline ready!")

In [ ]:
# P5 — Cell 6: Compare RAG vs Vanilla LLM on test questions
test_questions = [
    "When was ChatGPT launched and how quickly did it grow?",
    "What is the difference between GPT-3 and GPT-4?",
    "How does Microsoft use OpenAI's technology in its products?",
    "What is Google's Flan-T5 and why is it useful?",
    "What is RLHF and which company introduced it?",
    "What are Microsoft Phi models designed for?",
    "Who invented the Transformer architecture?",
]

print("=" * 75)
print("RAG vs VANILLA LLM COMPARISON")
print("=" * 75)

for q in test_questions:
    rag_ans, ctx = rag_answer(q)
    van_ans      = vanilla_answer(q)

    print(f"\n❓ {q}")
    print(f"\n   🤖 Vanilla : {van_ans}")
    print(f"   📚 RAG     : {rag_ans}")
    print(f"   📄 Source  : {ctx[0][:90]}...")
    print("-" * 75)


In [ ]:
# P5 — Cell 7: Interactive Q&A mode
# Ask your own questions about OpenAI, Google, or Microsoft!

print("🎯 Interactive RAG Q&A")
print("Type your question and press Enter. Type 'quit' to stop.")
print("Try: 'What models does OpenAI make?' or 'What is Vertex AI?'\n")

while True:
    user_q = input("Your question: ").strip()
    if not user_q or user_q.lower() in ['quit', 'exit', 'q']:
        print("Session ended.")
        break
    ans, ctx = rag_answer(user_q)
    print(f"\n✅ Answer    : {ans}")
    print(f"📄 Top source: {ctx[0][:100]}")
    print()


In [ ]:
# P5 — 🔥 Stretch Challenge: Add a PDF to the knowledge base
# Upload any PDF to Colab, then run this cell

!pip install PyPDF2 -q
import PyPDF2
import os

def load_pdf_into_rag(pdf_path, chunk_size=200):
    """Extract text from PDF, chunk it, add to FAISS index."""
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text() + " "

    # Split into ~200 word chunks
    words  = full_text.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    chunks = [c for c in chunks if len(c) > 50]  # filter very short chunks

    # Embed and add to FAISS
    new_embeddings = embedder.encode(chunks, convert_to_numpy=True).astype('float32')
    index.add(new_embeddings)
    documents.extend(chunks)

    print(f"✅ Added {len(chunks)} chunks from {os.path.basename(pdf_path)}")
    print(f"   Total documents in index: {index.ntotal}")
    return chunks

# USAGE — Upload a PDF to Colab, then:
# chunks = load_pdf_into_rag("/content/your_document.pdf")
# Then use rag_answer() as normal!

print("📋 PDF Loader ready!")
print("To use: upload a PDF to Colab and call:")
print('   chunks = load_pdf_into_rag("/content/your_file.pdf")')
print("Then ask questions about it using rag_answer()")

---
## 🎓 Session Summary

| Problem | What you built | Key takeaway |
|---------|---------------|-------------|
| P1 | Data cleaning + tokenization pipeline | Clean data > more data |
| P2 | GPT-2 fine-tuned on Indian recipes | Model style shifts after domain fine-tuning |
| P3 | LR sensitivity experiment | 5e-5 is the sweet spot for fine-tuning LLMs |
| P4 | BLEU & ROUGE evaluator | Precision (BLEU) + Recall (ROUGE) = full picture |
| P5 | Complete RAG pipeline | Retrieval grounds LLMs in facts, reduces hallucinations |

### 🚀 Next Steps
- **P2 → P4 connection:** Fine-tune Flan-T5 on DialogSum (from P1/P4 data), then re-evaluate BLEU/ROUGE. Compare base vs fine-tuned scores.
- **P5 upgrade:** Add W&B logging to any training loop: `import wandb; wandb.init(project="my-llm")`
- **Production RAG:** Swap FAISS for Pinecone, add reranking, measure answer quality with RAGAS library.
